# EDA of North Atlantic Basin part of ME4OH dataset - using SVD

Goal: To use SVD to reconstruct a low-rank approximation of OHC

In [ ]:
import numpy as np
import pandas as pd

from plotting_functions import plot_ohc_map

## Load data

In [ ]:
df = pd.read_csv("Data/ME4OH_EN411_OFAM3/NA_1993_2014.csv")

# Convert dates to datetime, save year month and year indication columns (aids plotting)
df['Date'] = pd.to_datetime(df['Date'])
df['year_month'] = df['Date'].dt.to_period('M')
df['year'] = df['Date'].dt.to_period('Y')
df

In [ ]:
# Filter for a single month
df = df[df["year_month"]=="1993-01"]
df

In [ ]:
# Create pivot table: rows = latitude, columns = longitude, values = OHC
matrix = df.pivot_table(index="Latitude", columns="Longitude", values="OHC")
matrix = matrix.fillna(0)  # Replace NaNs with 0
matrix

Thoughts: some of these latitudes/longitudes are on land and therefore it is unrealistic to have an OHC value here

In [ ]:
matrix.shape

In [ ]:
U, S, V = np.linalg.svd(matrix)
U.shape, S.shape, V.shape

In [ ]:
# Adjust matrix shapes to ensure U (m x n), S (n x n), V (n x n):
U = U[:, :S.shape[0]]  # Limit U to the number of singular values
S = np.diag(S)  # Convert S (a vector) into a diagonal matrix

In [ ]:
S

In [ ]:
r = 5  # ??
R = U[:, :r] @ S[:r, :r] @ V[:r, :]  # Use rank-reduced components for approximation 
R

In [ ]:
R.shape

In [ ]:
reconstructed_df = pd.DataFrame(R, index=matrix.index.to_list(), columns=matrix.columns.to_list())

In [ ]:
reconstructed_df = reconstructed_df.reset_index(drop=False, names='Latitude')
reconstructed_df

In [ ]:
unpivot_reconstruct = reconstructed_df.melt(id_vars='Latitude', var_name='Longitude', value_name='OHC')
unpivot_reconstruct

In [ ]:
plot_ohc_map(unpivot_reconstruct, "Reconstructed OHC", min_ohc=min(df.OHC), max_ohc=max(df.OHC), basin="NA")